In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [10]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [5]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [6]:
model_type = 'llama'
# model_type = 'qwen'

# MODEL_VERSION = '3'
MODEL_VERSION = '3.1'
# MODEL_VERSION = '3.3'

MODEL_SIZE = '8B'
# MODEL_SIZE = '70B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


In [ ]:
# ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']

In [23]:
# coef = 0.2
# coef = 0.25
# coef = 0.3
# coef = 0.4
# coef = 0.45
coef = 0.5
# coef = 0.55


max_tokens = 200

# prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'

# orig_lang = "english"
# orig_lang = "hindi"
# orig_lang = "german"
orig_lang = 'spanish'

# l = "german"
l = "hindi"
# l = "thai"
# l = 'italian'
# l = 'portuguese'

# l_controller = load_controller(llm, l, path=f'../all_gitignore/language_multi/')
l_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/language_multi/')

out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?
-----------------------------------------------------
Según las reglas de la FIFA, un balón de fútbol oficial debe tener un peso entre 410 y 450 gramos.

========================== + coef: 0.5, lang: hindi Control (normal) ==========================
¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?
-----------------------------------------------------
में फुटबॉल का उपयोग करने वाली फीफा में एक फुटबॉल का वजन लगभग 450-470 ग्राम होता है। यह वजन एक सामान्य फुटबॉल के वजन से थोड़ा अधिक होता है, जो कि 400-450 ग्राम होता है।

फ

Comparisions

In [21]:
lang2 = "hindi"
lang1 = "italian"
lang3 = "english"


l1_controller = load_controller_translate(llm, lang1, lang2, path=f'../all_gitignore/language_multi/')
l2_controller = load_controller_translate(llm, lang1, lang3, path=f'../all_gitignore/language_multi/')

l3_controller = load_controller_translate(llm, lang3, lang1, path=f'../all_gitignore/language_multi/')
l4_controller = load_controller_translate(llm, lang2, lang1, path=f'../all_gitignore/language_multi/')



Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

D

In [22]:
compare_cosine(l1_controller.directions, l2_controller.directions)

layer: -1, cosine: 0.3777655065059662
layer: -2, cosine: 0.3728252053260803
layer: -3, cosine: 0.3848298490047455
layer: -4, cosine: 0.41419360041618347
layer: -5, cosine: 0.4364524483680725
layer: -6, cosine: 0.37483206391334534
layer: -7, cosine: 0.2924160361289978
layer: -8, cosine: 0.33504343032836914
layer: -9, cosine: 0.4393605887889862
layer: -10, cosine: 0.4389570653438568
layer: -11, cosine: 0.45833808183670044
layer: -12, cosine: 0.4171558618545532
layer: -13, cosine: 0.4602075219154358
layer: -14, cosine: 0.4546092748641968
layer: -15, cosine: 0.43455761671066284
layer: -16, cosine: 0.2203870266675949
layer: -17, cosine: 0.3391492962837219
layer: -18, cosine: 0.2877112627029419
layer: -19, cosine: 0.37672093510627747
layer: -20, cosine: 0.3937227725982666
layer: -21, cosine: 0.38694682717323303
layer: -22, cosine: 0.34846439957618713
layer: -23, cosine: 0.3627947270870209
layer: -24, cosine: 0.4781742990016937
layer: -25, cosine: 0.46991828083992004
layer: -26, cosine: 0.576

0.417615897713169

In [23]:
compare_cosine(l3_controller.directions, l4_controller.directions)

layer: -1, cosine: 0.03683067858219147
layer: -2, cosine: -0.10372020304203033
layer: -3, cosine: -0.042768292129039764
layer: -4, cosine: 0.039587926119565964
layer: -5, cosine: 0.015600902959704399
layer: -6, cosine: 0.020478736609220505
layer: -7, cosine: -0.19523707032203674
layer: -8, cosine: -0.2240738421678543
layer: -9, cosine: -0.2050919234752655
layer: -10, cosine: -0.16165439784526825
layer: -11, cosine: 0.06776690483093262
layer: -12, cosine: -0.025502612814307213
layer: -13, cosine: -0.051080696284770966
layer: -14, cosine: -0.059766191989183426
layer: -15, cosine: -0.004298009444028139
layer: -16, cosine: 0.0592011883854866
layer: -17, cosine: 0.018897827714681625
layer: -18, cosine: 0.06311559677124023
layer: -19, cosine: 0.0029665473848581314
layer: -20, cosine: 0.16803759336471558
layer: -21, cosine: 0.33914434909820557
layer: -22, cosine: 0.10224539041519165
layer: -23, cosine: 0.021879443898797035
layer: -24, cosine: 0.02176785096526146
layer: -25, cosine: 0.03555804

0.024803640335918434